# Транскрипция совещаний (WhisperX large-v3 + pyannote 3.1)

Что делает блокнот:
1. Монтирует ваш Google Drive.
2. Берёт WAV-файл совещания (по умолчанию — самый свежий из `MeetingAssistant/Recordings/`).
3. Прогоняет его через WhisperX large-v3 + диаризацию pyannote-3.1.
4. (Опционально) GPT-4o-mini или GPT-4o правит метки спикеров.
5. Сохраняет результат в `MeetingAssistant/Transcripts/<тоже_имя>.txt` в формате `Speaker N: текст`.

**Перед первым запуском:**
- Убедитесь, что в Runtime → Change runtime type выбран **GPU (T4)**.
- Принимайте условия моделей pyannote один раз вручную:
  https://huggingface.co/pyannote/speaker-diarization-3.1 и https://huggingface.co/pyannote/segmentation-3.0
- Заполните `YOUR_HF_TOKEN` (для pyannote) и `OPENAI_API_KEY` (только если включаете cleanup) в ячейке «Настройки».

**Использование (каждый раз):**
Runtime → Run all. Через ~5–10 минут на 2-часовое совещание получите .txt в Drive.

## 0. Установка зависимостей

In [ ]:
!pip install -q whisperx
!pip uninstall -y -q numpy
!pip install -q "numpy<2.0.0"
!pip install -q openai-whisper openai

## 1. Настройки

Заполните токены и при желании измените режим cleanup и путь к WAV.

**Cleanup spikers** (правка меток `Speaker 0/1/...` через GPT):
- `"off"` — не трогать (рекомендуется для экономии токенов).
- `"mini"` — gpt-4o-mini (~$0.03 за 2 ч).
- `"4o"` — gpt-4o (~$0.50 за 2 ч, лучшее качество).

In [ ]:
# === Заполните ===
# Получите токен на https://huggingface.co/settings/tokens (тип "Read", галочка
# "Read access to contents of all public gated repos you can access")
YOUR_HF_TOKEN = ""    # вставьте сюда свой токен HuggingFace
OPENAI_API_KEY = ""   # нужен ТОЛЬКО если CLEANUP_MODE != 'off'

CLEANUP_MODE = "off"   # off / mini / 4o

# Папка, куда сохранять .txt (относительно MyDrive)
TRANSCRIPTS_DIR = "Programming/MeetingAssistant/Transcripts"

# Путь к конкретному WAV (относительно MyDrive, БЕЗ '/content/drive/MyDrive/').
# Если задан - используется он.
# Пример (как вы только что загружали в приложении):
#   TARGET_WAV_PATH = "DOC/Poland/Unitrank/Meetings/1 meeting/Запись/504180e6_2026-04-23_12-42-36_meeting_504180e6_part034.wav"
TARGET_WAV_PATH = ""

# Папка для поиска свежего WAV, если TARGET_WAV_PATH пуст (относительно MyDrive)
RECORDINGS_DIR = "Programming/MeetingAssistant/Recordings"


## 2. Подключить Drive и найти WAV

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')

tr_dir = DRIVE_ROOT / TRANSCRIPTS_DIR
tr_dir.mkdir(parents=True, exist_ok=True)

if TARGET_WAV_PATH:
    # Очистить от ведущих слешей и обёрток типа '/content/drive/MyDrive/...'
    rel = TARGET_WAV_PATH.strip()
    for prefix in ('/content/drive/MyDrive/', 'content/drive/MyDrive/', 'MyDrive/'):
        if rel.startswith(prefix):
            rel = rel[len(prefix):]
            break
    rel = rel.lstrip('/').lstrip('\\')
    wav_path = DRIVE_ROOT / rel
    if not wav_path.exists():
        raise FileNotFoundError(
            f"Не найден WAV: {wav_path}\n"
            f"Проверьте TARGET_WAV_PATH (он указывается относительно MyDrive)."
        )
else:
    rec_dir = DRIVE_ROOT / RECORDINGS_DIR
    if not rec_dir.exists():
        raise FileNotFoundError(
            f"Папка не существует: {rec_dir}\n"
            f"Либо запустите запись в приложении (создаст папку и положит WAV), "
            f"либо укажите конкретный файл через TARGET_WAV_PATH."
        )
    wavs = sorted(rec_dir.glob('*.wav'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not wavs:
        # Подсказка: показать что есть в папке
        all_items = list(rec_dir.iterdir())
        listing = '\n  '.join(str(p.name) for p in all_items[:20]) or '(пусто)'
        raise FileNotFoundError(
            f"В {rec_dir} нет .wav файлов.\n"
            f"Содержимое папки:\n  {listing}\n\n"
            f"Если ваш WAV лежит в другом месте Drive - укажите его через TARGET_WAV_PATH "
            f"в ячейке Настройки. Например:\n"
            f"  TARGET_WAV_PATH = \"DOC/Poland/Unitrank/Meetings/1 meeting/Запись/имя_файла.wav\""
        )
    wav_path = wavs[0]
    print(f"TARGET_WAV_PATH пуст - взял самый свежий WAV из {rec_dir}")

print(f"Будет обработан: {wav_path}")
print(f"Размер: {wav_path.stat().st_size / (1024*1024):.1f} MB")

out_txt = tr_dir / (wav_path.stem + '.txt')
print(f"Результат запишется в: {out_txt}")


## 3. WhisperX: транскрипция + диаризация

In [ ]:
import whisperx
import gc
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
print(f"device={device}, compute_type={compute_type}")

print("Загрузка модели large-v3...")
model = whisperx.load_model("large-v3", device, compute_type=compute_type)

print("Загрузка аудио...")
audio = whisperx.load_audio(str(wav_path))

print("Транскрипция...")
result = model.transcribe(audio, batch_size=16)
print(f"Язык: {result.get('language')}, сегментов: {len(result.get('segments', []))}")

# Освободить память перед диаризацией
del model
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
from whisperx.diarize import DiarizationPipeline

print("Диаризация (pyannote 3.1)...")
diarize_model = DiarizationPipeline(
    model_name="pyannote/speaker-diarization-3.1",
    use_auth_token=YOUR_HF_TOKEN,
    device=device,
)
diarize_segments = diarize_model(audio)
result = whisperx.assign_word_speakers(diarize_segments, result)
print("Диаризация готова.")

## 4. Сборка диалога `Speaker N: текст`

In [ ]:
lines = []
for seg in result.get('segments', []):
    speaker = seg.get('speaker', 'Unknown')
    text = (seg.get('text') or '').strip()
    if text:
        lines.append(f"{speaker}: {text}")

dialogue = "\n".join(lines)
print(f"Получено {len(lines)} реплик, {len(dialogue)} символов")
print('---')
print(dialogue[:1500])
print('---')

## 5. (Опционально) Cleanup спикеров через GPT

Запускается, только если `CLEANUP_MODE in {"mini", "4o"}`. Иначе ячейка ничего не делает.

In [ ]:
if CLEANUP_MODE in ("mini", "4o"):
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY не задан, а CLEANUP_MODE требует OpenAI.")

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    cleanup_model = "gpt-4o-mini" if CLEANUP_MODE == "mini" else "gpt-4o"

    prompt = (
        "В этом транскрипте могут быть ошибки разделения спикеров (диаризация иногда\n"
        "путает соседние реплики). Поправь только метки спикеров (Speaker 0/1/...),\n"
        "опираясь на логику диалога (вопрос/ответ, обращения по имени и т.п.).\n"
        "НЕ меняй текст реплик. Верни весь диалог в том же формате 'Speaker N: текст',\n"
        "одна реплика на строку.\n\n"
        f"Транскрипт:\n{dialogue}\n"
    )

    print(f"Cleanup через {cleanup_model}...")
    resp = client.chat.completions.create(
        model=cleanup_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    cleaned = resp.choices[0].message.content.strip()
    print(f"Получено {len(cleaned)} символов от {cleanup_model}")
    dialogue = cleaned
else:
    print("Cleanup пропущен (CLEANUP_MODE='off')")

## 6. Сохранение в Drive

In [ ]:
with open(out_txt, 'w', encoding='utf-8') as f:
    f.write(dialogue)

print(f"Готово! Транскрипт сохранён:\n{out_txt}")
print(f"Размер: {out_txt.stat().st_size} байт")
print("\nGoogle Drive Desktop синхронизирует этот файл к вам на компьютер за 30-60 сек.")
print("Дальше в приложении: '📝 Загрузить транскрипт' -> выберите этот .txt -> 'ОТЧЕТ'.")